In [1]:
# code to segment water in stream

from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from PIL import Image
import torch

image_path = "copied_files/DSCF0583.JPG"
#image_path = "livestream-snapshots/UNAVAILABLE.jpg"

processor = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")
model = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined")

image = Image.open(image_path).convert("RGB")
texts = ["water"]
inputs = processor(text=texts, images=[image] * len(texts), padding=True, return_tensors="pt")

outputs = model(**inputs)
logits = outputs.logits
probs = torch.sigmoid(logits)
masks = (probs > 0.5).int()
water_pixels = (masks == 1).sum().item()

print(f"{water_pixels} Pixels are wet!")

print(logits.shape)
#torch.Size([# of text inputs, 352, 352])

/home/jans26/.conda/envs/streamflow-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/home/jans26/.conda/envs/streamflow-env/lib/python3.13/site-packages/transformers/image_processing_utils.py:44: UserWarning: The following named arguments are not valid for `ViTImageProcessor.preprocess` and were ignored: 'padding'
  return self.preprocess(images, **kwargs)


41275 Pixels are wet!
torch.Size([1, 352, 352])


In [2]:
# code to visualize segmented image

import cv2
import numpy as np

#image_path = "test_image.JPG"
#image_path = "livestream-snapshots/UNAVAILABLE.jpg"

# Assume original image is loaded as a NumPy array, shape (352, 352, 3)
image = cv2.imread(image_path)
image = cv2.resize(image, (352, 352))  # ensure it's the same size

# Convert mask[0] (PyTorch tensor) to NumPy uint8
mask_water = (masks[0].cpu().numpy() * 255).astype(np.uint8)

# Make colored overlay (e.g. red)
colored_mask = np.zeros_like(image)
colored_mask[:, :, 1] = mask_water  # Red channel

# Blend overlay with image
overlay = cv2.addWeighted(image, 0.7, colored_mask, 0.3, 0)

cv2.imwrite("overlayed_image.jpg", overlay)


True

In [17]:
# code to get height of the measuring tape segmentation in pixels

tape_mask = masks[0]  # binary mask for "measuring tape", shape: [352, 352]

# Get the row indices where the mask is 1
rows_with_mask = torch.any(tape_mask == 1, dim=1).nonzero(as_tuple=True)[0]

if len(rows_with_mask) > 0:
    height_pixels = rows_with_mask[-1] - rows_with_mask[0] + 1
    lowest_row = rows_with_mask[-1].item()  
    print(f"Measuring tape segmentation height: {height_pixels.item()} pixels")
    print(f"Lowest row (bottom of measuring tape): {lowest_row} (y-coordinate)")
else:
    print("No measuring tape detected in the mask.")


Measuring tape segmentation height: 190 pixels
Lowest row (bottom of measuring tape): 235 (y-coordinate)
